# Level 2: Intermediate Techniques - CIFAR-10 Classification

## Objective
Improve performance with advanced data augmentation, regularization, and hyperparameter tuning.

## Target Accuracy: ≥90%

## Deliverables:
- ✅ Augmentation pipeline
- ✅ Ablation study (with/without augmentation)
- ✅ Accuracy comparison table
- ✅ Analysis document

---

In [ ]:
# Install required packages
# !pip install torch torchvision matplotlib seaborn tqdm albumentations

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import os

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Advanced Data Augmentation Pipeline

Using Albumentations library for powerful augmentations.

In [ ]:
# CIFAR-10 class names
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
           'dog', 'frog', 'horse', 'ship', 'truck']
NUM_CLASSES = 10

# ImageNet stats for normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Advanced augmentation pipeline using Albumentations
train_augmentation = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, 
                    min_holes=1, min_height=8, min_width=8, p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
    A.GaussNoise(var_limit=(10, 50), p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2()
])

# Basic augmentation (for ablation study)
basic_augmentation = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2()
])

# No augmentation (baseline)
no_augmentation = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2()
])

# Test transform
test_augmentation = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2()
])

print("Augmentation Pipelines Created:")
print("1. No Augmentation (baseline)")
print("2. Basic Augmentation (HorizontalFlip only)")
print("3. Advanced Augmentation (Full pipeline)")

In [ ]:
# Custom Dataset class for Albumentations
class CIFAR10Albumentations(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        image = np.array(image)
        
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        
        return image, label

# Load raw CIFAR-10 (no transforms)
raw_train_dataset = datasets.CIFAR10(root='./data', train=True, download=True)
raw_test_dataset = datasets.CIFAR10(root='./data', train=False, download=True)

# Create train/val split
train_size = 40000
val_size = 10000

indices = list(range(len(raw_train_dataset)))
np.random.shuffle(indices)
train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]

print(f"Train: {len(train_indices)}, Val: {len(val_indices)}, Test: {len(raw_test_dataset)}")

## 2. Visualize Augmentations

In [ ]:
def visualize_augmentations(dataset, idx, transform, num_augments=8):
    """Visualize different augmentations of the same image"""
    image, label = dataset[idx]
    image = np.array(image)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f'Augmentation Examples - Class: {CLASSES[label]}', fontsize=16)
    
    for i, ax in enumerate(axes.flat):
        if i == 0:
            ax.imshow(image)
            ax.set_title('Original', fontsize=12)
        else:
            augmented = transform(image=image)
            aug_img = augmented['image']
            # Denormalize for visualization
            aug_img = aug_img.permute(1, 2, 0).numpy()
            aug_img = aug_img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
            aug_img = np.clip(aug_img, 0, 1)
            ax.imshow(aug_img)
            ax.set_title(f'Augmented {i}', fontsize=12)
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('results/augmentation_examples.png', dpi=150, bbox_inches='tight')
    plt.show()

# Visualize augmentations
visualize_augmentations(raw_train_dataset, 100, train_augmentation)

## 3. Model Architecture with Regularization

In [ ]:
def create_improved_resnet50(num_classes=10, dropout_rate=0.5):
    """Create ResNet50 with improved regularization"""
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    
    # Replace final FC layer with more regularization
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout_rate),
        nn.Linear(num_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(dropout_rate * 0.6),
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(dropout_rate * 0.4),
        nn.Linear(256, num_classes)
    )
    
    return model

print("Model architecture created with improved regularization")

## 4. Training Functions

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{running_loss/total:.4f}', 
                         'acc': f'{100.*correct/total:.2f}%'})
    
    return running_loss / len(train_loader), 100. * correct / total


def evaluate(model, data_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc='Evaluating'):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return running_loss / len(data_loader), 100. * correct / total, np.array(all_preds), np.array(all_labels)

In [ ]:
def train_model(augmentation_type, train_transform, num_epochs=20):
    """Train model with specified augmentation"""
    print(f"\n{'='*60}")
    print(f"Training with: {augmentation_type}")
    print(f"{'='*60}")
    
    # Create datasets
    train_subset = Subset(raw_train_dataset, train_indices)
    val_subset = Subset(raw_train_dataset, val_indices)
    
    train_dataset = CIFAR10Albumentations(train_subset, train_transform)
    val_dataset = CIFAR10Albumentations(val_subset, test_augmentation)
    test_dataset = CIFAR10Albumentations(raw_test_dataset, test_augmentation)
    
    # Create loaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
    
    # Create model
    model = create_improved_resnet50(NUM_CLASSES).to(device)
    
    # Training config
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # Label smoothing
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    # Training history
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    best_model_state = None
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        scheduler.step()
        
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            print(f"*** New best: {val_acc:.2f}% ***")
    
    # Test evaluation
    model.load_state_dict(best_model_state)
    test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
    
    print(f"\n{augmentation_type} - Test Accuracy: {test_acc:.2f}%")
    
    return {
        'name': augmentation_type,
        'history': history,
        'best_val_acc': best_val_acc,
        'test_acc': test_acc,
        'model_state': best_model_state,
        'test_preds': test_preds,
        'test_labels': test_labels
    }

## 5. Ablation Study - Train with Different Augmentations

In [ ]:
# Run ablation study
results = {}

# Train with no augmentation
results['no_aug'] = train_model('No Augmentation', no_augmentation, num_epochs=15)

# Train with basic augmentation
results['basic_aug'] = train_model('Basic Augmentation', basic_augmentation, num_epochs=15)

# Train with advanced augmentation
results['advanced_aug'] = train_model('Advanced Augmentation', train_augmentation, num_epochs=20)

## 6. Ablation Study Results and Comparison

In [ ]:
# Create comparison table
print("\n" + "="*70)
print("ABLATION STUDY RESULTS")
print("="*70)
print(f"{'Augmentation Type':<25} {'Val Acc (%)':<15} {'Test Acc (%)':<15} {'Improvement'}")
print("-"*70)

baseline_acc = results['no_aug']['test_acc']
for key, res in results.items():
    improvement = res['test_acc'] - baseline_acc
    imp_str = f"+{improvement:.2f}%" if improvement > 0 else f"{improvement:.2f}%"
    print(f"{res['name']:<25} {res['best_val_acc']:<15.2f} {res['test_acc']:<15.2f} {imp_str}")

print("="*70)

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {'no_aug': 'red', 'basic_aug': 'blue', 'advanced_aug': 'green'}
labels = {'no_aug': 'No Augmentation', 'basic_aug': 'Basic', 'advanced_aug': 'Advanced'}

# Training loss comparison
for key, res in results.items():
    epochs = range(1, len(res['history']['train_loss']) + 1)
    axes[0].plot(epochs, res['history']['train_loss'], color=colors[key], 
                 label=labels[key], linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation accuracy comparison
for key, res in results.items():
    epochs = range(1, len(res['history']['val_acc']) + 1)
    axes[1].plot(epochs, res['history']['val_acc'], color=colors[key], 
                 label=labels[key], linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Validation Accuracy Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Test accuracy bar chart
names = [res['name'] for res in results.values()]
test_accs = [res['test_acc'] for res in results.values()]
bars = axes[2].bar(names, test_accs, color=['red', 'blue', 'green'], edgecolor='black')
axes[2].axhline(y=90, color='orange', linestyle='--', label='Target (90%)')
axes[2].set_ylabel('Test Accuracy (%)')
axes[2].set_title('Test Accuracy Comparison')
axes[2].legend()
for bar, acc in zip(bars, test_accs):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{acc:.2f}%', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig('results/ablation_study_level2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save best model (advanced augmentation)
best_result = results['advanced_aug']
torch.save(best_result['model_state'], 'models/resnet50_cifar10_level2.pth')
print(f"Best model saved: Advanced Augmentation with {best_result['test_acc']:.2f}% test accuracy")

## 7. Confusion Matrix for Best Model

In [ ]:
# Confusion matrix for best model
best = results['advanced_aug']
cm = confusion_matrix(best['test_labels'], best['test_preds'])

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix - Level 2 (Test Acc: {best["test_acc"]:.2f}%)')
plt.tight_layout()
plt.savefig('results/confusion_matrix_level2.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nClassification Report:")
print(classification_report(best['test_labels'], best['test_preds'], target_names=CLASSES))

## 8. Summary and Analysis

### Key Findings from Ablation Study:

1. **No Augmentation**: Baseline performance, prone to overfitting
2. **Basic Augmentation**: Small improvement with just horizontal flips
3. **Advanced Augmentation**: Best performance with comprehensive augmentation

### Techniques Used:
- **Data Augmentation**: HorizontalFlip, ColorJitter, ShiftScaleRotate, CoarseDropout, GaussNoise
- **Regularization**: Dropout, BatchNorm, Weight Decay, Label Smoothing
- **Optimization**: AdamW optimizer, Cosine Annealing LR scheduler

### Analysis:
- Augmentation significantly reduces overfitting
- CoarseDropout (Cutout) helps model learn more robust features
- Label smoothing prevents overconfident predictions

In [ ]:
# Final Summary
print("="*60)
print("LEVEL 2 SUMMARY - INTERMEDIATE TECHNIQUES")
print("="*60)
print(f"\nBest Model: Advanced Augmentation")
print(f"Test Accuracy: {results['advanced_aug']['test_acc']:.2f}%")
print(f"\nTarget Accuracy: ≥90%")
status = '✅ PASSED' if results['advanced_aug']['test_acc'] >= 90 else '❌ NOT PASSED'
print(f"Status: {status}")
print(f"\nImprovement over baseline: +{results['advanced_aug']['test_acc'] - results['no_aug']['test_acc']:.2f}%")
print("="*60)